# P80 — Modelización estadística: las dos culturas

## 1. Título y paper

**Paper:** *Statistical Modeling: The Two Cultures*  
**Autoría:** Leo Breiman  
**Año y venue:** 2001 · Statistical Science, 16(3), 199–231  
**Nivel:** L1 · **Motor:** `dos_culturas`  
**Ficha completa:** [`P80_dos_culturas`](../../papers/foundational/P80_dos_culturas/README.md)

**Hito:** Nombra la división que organiza el campo: suponer un mecanismo generador frente a medir la capacidad de predecir.

- [doi:10.1214/ss/1009213726](https://doi.org/10.1214/ss/1009213726)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La estadística académica suponía que los datos venían de un modelo con forma conocida y juzgaba los métodos por el ajuste a ese supuesto. Si el supuesto es falso —y casi siempre lo es— las conclusiones sobre el mecanismo no valen nada.
2. Ejecutar una implementación mínima de la propuesta: Distinguir dos culturas y sus criterios: la del modelo de datos, que valida supuestos, y la algorítmica, que trata el mecanismo como desconocido y se juzga por exactitud predictiva medida fuera de muestra.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P76
- P79
- P75


## 4. Intuición

Hay dos formas de usar un modelo. Una supone que los datos salen de un mecanismo con forma conocida y usa el modelo para describirlo. La otra trata el mecanismo como desconocido y solo pregunta si predice. Breiman sostiene que la primera se ha equivocado durante décadas.


## 5. Concepto mínimo

```text
Cultura del modelo de datos     Cultura algorítmica
────────────────────────────    ─────────────────────────
supone la forma del mecanismo    trata el mecanismo como desconocido
valida supuestos                 mide exactitud FUERA DE MUESTRA
interpreta coeficientes          acepta modelos difíciles de leer

Efecto Rashomon: muchos modelos con exactitud casi igual
                 y explicaciones incompatibles entre sí
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('dos_culturas', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánto acierta el modelo lineal si el mecanismo real es una interacción?
2. ¿Y añadiendo el término de interacción?
3. ¿Cuántos modelos distintos alcanzan una exactitud parecida?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('dos_culturas', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('dos_culturas', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El modelo lineal en las variables originales llega a **0,70** y con el término de interacción, a **0,90**. Y cuatro modelos distintos alcanzan exactitudes entre 0,8625 y 0,9 —una banda de 0,0375— con **coeficientes muy distintos** para la primera variable y para su copia ruidosa.


## 10. Comentario pedagógico

Ese es el efecto Rashomon y es el argumento más incómodo del artículo: si varios modelos casi equivalentes cuentan historias distintas sobre qué variable importa, entonces la historia no está determinada por los datos. Interpretar los coeficientes de uno de ellos como «el efecto» de cada variable es elegir una narración entre varias compatibles.


## 11. Error o anti-patrón deliberado

Anti-patrón: interpretar coeficientes de un modelo cuya forma no se ha validado.


In [ ]:
print('Si el mecanismo real es x1*x2 y ajustas un modelo lineal en x1 y x2,')
print('los coeficientes NO son «el efecto de cada variable»: son el mejor')
print('ajuste lineal a algo que no es lineal. Describen un mecanismo inexistente.')

## 12. Corrección

La separación que propone Breiman entre los dos objetivos:


In [ ]:
r = run_paper_lab('dos_culturas', seed=7)['result']
print('mecanismo real  :', r['mecanismo_real'])
print('modelo de datos :', r['cultura_del_modelo_de_datos']['exactitud_prueba'])
print('algoritmico     :', r['cultura_algoritmica']['exactitud_prueba'])
for m in r['efecto_rashomon']:
    print('  rashomon:', m)

## 13. Desafío guiado

Compara los coeficientes de x1 y de x4 —su copia ruidosa— entre los modelos Rashomon, y explica por qué un ranking de importancia de variables puede ser inestable.


In [ ]:
r = run_paper_lab('dos_culturas', seed=3)['result']
show(r)

## 14. Desafío autónomo

Ajusta a un mismo conjunto tres modelos de familias distintas con exactitudes parecidas y compara sus explicaciones de qué variable importa. Documenta si coinciden.


## 15. Evidencia de aprendizaje

Guarda la banda de exactitud de los modelos Rashomon con sus coeficientes, y tu criterio para decidir cuándo la interpretación de un coeficiente está justificada.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P80_dos_culturas/README.md) · evaluación formal: [`assessments/papers/P80_dos_culturas.md`](../../assessments/papers/P80_dos_culturas.md)


## 16. Cierre

Si el modelo se juzga por predecir, hay que saber qué variables darle. Y elegirlas de una en una falla de dos formas distintas.


## 17. Conexión con el siguiente hito

- P82
- P62

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
